# VoiceGuard — train on Kaggle (P100)

Full pipeline: HF data → content-matched MMS-TTS fakes → manifests → (RawBoost) →
fine-tune wav2vec2 + AASIST end to end → evaluate per dataset/language.
Output: `/kaggle/working/aasist_indicw2v.pt` — download it and drop into `backend/models/`.

**Settings → Accelerator → GPU P100** (needs phone verification once).
**Save Version → Save & Run All (Commit)** to run in the background across the 12 h limit.
Raw data goes to `/kaggle/temp` (ephemeral); the checkpoint to `/kaggle/working` (kept).

In [ ]:
import torch, os
assert torch.cuda.is_available(), 'Enable GPU in Settings → Accelerator'
print(torch.cuda.get_device_name(0))
!pip -q install -U 'transformers>=4.44' 'datasets>=2.20' huggingface_hub soundfile librosa pyyaml scipy

In [ ]:
REPO_URL = ''   # <-- your GitHub clone URL. Or add the repo as a Kaggle Dataset and adjust below.
HF_TOKEN = ''   # <-- optional: enables ai4bharat/indicwav2vec-hindi (accept its licence first)

%cd /kaggle/working
if REPO_URL:
    !rm -rf voiceguard && git clone --depth 1 {REPO_URL} voiceguard
elif os.path.isdir('/kaggle/input'):
    src = next((f'/kaggle/input/{d}' for d in os.listdir('/kaggle/input') if os.path.isdir(f'/kaggle/input/{d}/backend')), None)
    assert src, 'Set REPO_URL or attach the repo as a Dataset'
    !cp -r {src} voiceguard
%cd voiceguard
os.makedirs('/kaggle/temp/vgdata', exist_ok=True)
if os.path.islink('data') or os.path.isdir('data'): pass
!rm -rf data && ln -s /kaggle/temp/vgdata data
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
!ls

In [ ]:
# --- knobs ---
FRONTEND = 'facebook/wav2vec2-xls-r-300m'   # or ai4bharat/indicwav2vec-hindi (needs HF_TOKEN)
EPOCHS   = 20
UNFREEZE = 6        # frontend frozen for N epochs then fine-tuned (the quality lever)
LOSS     = 'oc_softmax'
FAKE_N   = 1200
LIMIT    = None     # e.g. 400 for a smoke run first

import yaml, pathlib
p = pathlib.Path('training/config_train.yaml'); cfg = yaml.safe_load(p.read_text())
cfg['device'] = 'cuda'
cfg['frontend'].update(model_id=FRONTEND, layer=-1, stage2_unfreeze_epoch=UNFREEZE)
cfg['loss']['name'] = LOSS
cfg['epochs'] = EPOCHS
cfg['num_workers'] = 2
cfg['checkpoint']['out'] = '/kaggle/working/aasist_indicw2v.pt'
p.write_text(yaml.safe_dump(cfg, sort_keys=False)); print(yaml.safe_dump(cfg, sort_keys=False))

In [ ]:
args = f'--config training/config_train.yaml --fake-n {FAKE_N} --epochs {EPOCHS}'
if LIMIT: args += f' --limit {LIMIT}'
!python -m training.pipeline_run {args}

In [ ]:
# checkpoint is at /kaggle/working/aasist_indicw2v.pt -> download from the Output tab after commit.
!ls -la /kaggle/working/*.pt && python - <<'PY'
import torch; c = torch.load('/kaggle/working/aasist_indicw2v.pt', map_location='cpu', weights_only=False)
print('dev_eer', c.get('dev_eer'), 'epoch', c.get('epoch'), 'frontend', c.get('frontend_model_id'),
      'finetuned' , c.get('frontend') is not None)
PY